<div style='text-align: center; padding: 30px; background: linear-gradient(135deg, #1e0b36 0%, #4c1d95 100%); border-radius: 15px; margin: 10px 0; box-shadow: 0 10px 30px rgba(0,0,0,0.3);'>
  <h1 style='color: white; margin: 0 0 8px 0; font-size: 2.5em;'>🎙️ SeedVC Voice Changer & Singing Cloner</h1>
  <h3 style='color: #c084fc; margin: 0 0 5px 0; font-weight: 400;'>Colab Free Tier (T4 GPU) — Created by <strong>AIQUEST Academy</strong></h3>
  <p style='color: #e9d5ff; margin: 0; text-align: center;'>Zero-Shot Voice Conversion • Vocal Demixing • Song Cover Cloning | Wan2GP Engine</p>
</div>

---

<div align="center">
  <img src="https://img.shields.io/badge/AIQUESTAcademy-blueviolet?style=for-the-badge&logo=youtube&logoColor=white" />
  <img src="https://img.shields.io/badge/Colab-T4%20GPU-4285F4?style=for-the-badge&logo=google-colab&logoColor=white" />
  <br>
  <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1">
    <img src="https://img.shields.io/badge/Subscribe%20on%20YouTube-FF0000?style=for-the-badge&logo=youtube&logoColor=white" />
  </a>
  &nbsp;
  <a href="https://x.com/aiquestacademy">
    <img src="https://img.shields.io/badge/Follow%20on%20X-000000?style=for-the-badge&logo=x&logoColor=white" />
  </a>
</div>

---

### Features
1. **Zero-Shot Voice Conversion**: Convert any source speech or singing audio into a target speaker's voice using only a short reference voice sample.
2. **Vocal & Background Separation (Demixing)**: Automatically isolates vocals from background tracks using RoFormer, converts the vocals, and mixes the original music/backing tracks back.
3. **Multiple Model Engines**:
   - **v1.0 Speech**: Optimized for spoken dialogue conversion.
   - **v1.0 Singing**: 44.1kHz sample rate with pitch tracking (RMVPE) to preserve vocal melodies.
   - **v2 Speech**: Accent/Prosody transfer using ASTRAL auto-regressive flow matching.
4. **Memory Efficient**: Uses `mmgp` Profile 4 dynamic offloading to fit within Tesla T4 VRAM.

---

### Quick Start
1. Set **Runtime → Change runtime type → T4 GPU** in the top Colab menu.
2. Run all cells in order.
3. Access the public Gradio link generated at the end to convert your voice or songs!


In [ ]:
#@title 🖥️ Step 1: Environment Setup
# Drops cache files and optimizes GPU memory allocations for Colab's RAM limits.
import os, gc, psutil

print('=== Colab T4 Environment Setup ===')
print(f'RAM: {psutil.virtual_memory().total / 1024**3:.1f} GB total, {psutil.virtual_memory().available / 1024**3:.1f} GB available')

os.system('echo 3 | sudo tee /proc/sys/vm/drop_caches > /dev/null 2>&1')
os.system('echo 1 | sudo tee /proc/sys/vm/overcommit_memory > /dev/null 2>&1')
gc.collect()

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True,garbage_collection_threshold:0.6'
os.environ['MALLOC_TRIM_THRESHOLD_'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

print('✅ Environment optimized!')

In [ ]:
#@title 📦 Step 2: Clone Wan2GP & Install Dependencies
import subprocess
try:
    subprocess.run(['nvidia-smi'], check=True)
    print('GPU Active!')
except Exception:
    print('WARNING: No GPU active. Please go to Runtime → Change runtime type and select T4 GPU.')

!git clone https://github.com/DeepBeepMeep/Wan2GP.git

# Install requirements (preserving Colab's pre-installed optimized PyTorch stack)
!pip install --timeout 120 --retries 5 -q -r Wan2GP/requirements.txt
!pip install --timeout 120 --retries 5 -q mmgp gradio soundfile audio-separator[gpu]

print('✅ Dependencies installed successfully!')

In [ ]:
#@title 📥 Step 3: Download SeedVC & RoFormer Models
# Downloads the required model weights to /content/tmp_models and symlinks them to prevent storage OOMs.
import os
os.environ['HF_HOME'] = '/content/tmp_models/hf_home'
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'

from huggingface_hub import hf_hub_download

COMPANION_REPO = 'DeepBeepMeep/LTX-2'
WAN_REPO = 'DeepBeepMeep/Wan2.1'
MODEL_DIR = 'Wan2GP/models'
CKPT_DIR = 'Wan2GP/ckpts'
TMP_DIR = '/content/tmp_models'

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(TMP_DIR, exist_ok=True)

def download_and_link(repo, filename, subfolder=None, target_base_dir=MODEL_DIR):
    if subfolder:
        huggingface_filename = f"{subfolder}/{filename}"
        dest_dir = os.path.join(target_base_dir, subfolder)
        tmp_dest_dir = os.path.join(TMP_DIR, subfolder)
    else:
        huggingface_filename = filename
        dest_dir = target_base_dir
        tmp_dest_dir = TMP_DIR

    os.makedirs(dest_dir, exist_ok=True)
    os.makedirs(tmp_dest_dir, exist_ok=True)

    dest_path = os.path.join(dest_dir, filename)
    actual_path = os.path.join(TMP_DIR, huggingface_filename)

    # Ensure nested subdirectories are created (e.g., seed-vc/v2)
    os.makedirs(os.path.dirname(dest_path), exist_ok=True)
    os.makedirs(os.path.dirname(actual_path), exist_ok=True)

    if os.path.exists(dest_path):
        print(f"  ✓ Already exists: {os.path.join(subfolder or '', filename)}")
        return

    print(f"Downloading {huggingface_filename} → /content/tmp_models...")
    hf_hub_download(repo_id=repo, filename=huggingface_filename, local_dir=TMP_DIR, local_dir_use_symlinks=False)

    os.symlink(actual_path, dest_path)
    print(f"  ✓ {os.path.join(subfolder or '', filename)} (symlinked)")

# 1. SeedVC Core Checkpoints (Speech, Singing, Accent, and RMVPE pitch estimator)
SEEDVC_FILES = [
    'DiT_seed_v2_uvit_whisper_small_wavenet_bigvgan_pruned.pth',
    'config_dit_mel_seed_uvit_whisper_small_wavenet.yml',
    'DiT_seed_v2_uvit_whisper_base_f0_44k_bigvgan_pruned_ft_ema_v2.pth',
    'config_dit_mel_seed_uvit_whisper_base_f0_44k.yml',
    'campplus_cn_common.bin',
    'rmvpe.pt',
    'v2/ar_base.pth',
    'v2/cfm_small.pth',
    'bsq32/bsq32_light.pth',
    'bsq2048/bsq2048_light.pth',
]
for f in SEEDVC_FILES:
    download_and_link(COMPANION_REPO, f, 'seed-vc')

# 2. BigVGAN Vocoders (22kHz & 44kHz)
BIGVGAN_FILES = ['config.json', 'bigvgan_generator.pt']
for f in BIGVGAN_FILES:
    download_and_link(COMPANION_REPO, f, 'bigvgan_v2_22khz_80band_256x')
    download_and_link(COMPANION_REPO, f, 'bigvgan_v2_44khz_128band_512x')

# 3. Whisper Small semantic tokenizer
WHISPER_SMALL_FILES = [
    'added_tokens.json', 'config.json', 'generation_config.json', 'merges.txt',
    'model.safetensors', 'normalizer.json', 'preprocessor_config.json',
    'special_tokens_map.json', 'tokenizer.json', 'tokenizer_config.json', 'vocab.json',
]
for f in WHISPER_SMALL_FILES:
    download_and_link(COMPANION_REPO, f, 'whisper-small')

# 4. HuBERT content extractor (for Mode 3 Accent converter)
HUBERT_FILES = ['config.json', 'preprocessor_config.json', 'pytorch_model.bin']
for f in HUBERT_FILES:
    download_and_link(COMPANION_REPO, f, 'hubert-large-ll60k')

# 5. RoFormer Vocal/Background demixer checkpoints
ROFORMER_FILES = [
    'model_bs_roformer_ep_317_sdr_12.9755.ckpt',
    'model_bs_roformer_ep_317_sdr_12.9755.yaml',
    'download_checks.json'
]
for f in ROFORMER_FILES:
    download_and_link(WAN_REPO, f, 'roformer', target_base_dir=CKPT_DIR)

# Cleanup Hugging Face cache to save disk space
import shutil
for d in [os.path.join(MODEL_DIR, '.cache'), os.path.join(TMP_DIR, '.cache'), '/content/tmp_models/hf_home']:
    if os.path.exists(d):
        try:
            shutil.rmtree(d)
        except Exception:
            pass

os.system('df -h /content')
print('\n✅ All downloads and model symlinks successfully created!')

In [ ]:
#@title ⚙️ Step 4: Write the Voice Changer Script
# Generates run_seedvc_changer.py to handle Gradio routing, audio processing, and demixing.
%%writefile run_seedvc_changer.py
import gc
import os
import sys
import tempfile
import traceback
import numpy as np
import soundfile as sf
import psutil

# ---- bootstrap Wan2GP ----
WAN2GP_DIR = os.path.abspath("Wan2GP")
sys.path.insert(0, WAN2GP_DIR)
os.chdir(WAN2GP_DIR)

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:128,garbage_collection_threshold:0.5"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUDA_LAUNCH_BLOCKING"] = "0"

import torch
import torchaudio

# ---- Patch torchaudio to bypass torchcodec loading issues ----
def patched_torchaudio_load(filepath, frame_offset=0, num_frames=-1, normalize=True, channels_first=True, format=None, buffer_size=4096):
    data, samplerate = sf.read(filepath, dtype='float32')
    tensor = torch.from_numpy(data)
    if channels_first:
        if tensor.ndim == 1:
            tensor = tensor.unsqueeze(0)
        else:
            tensor = tensor.T
    return tensor, samplerate

def patched_torchaudio_save(filepath, src, sample_rate, channels_first=True, format=None, encoding=None, bits_per_sample=None, buffer_size=4096):
    if torch.is_tensor(src):
        src_np = src.detach().cpu().numpy()
    else:
        src_np = src
    if channels_first and src_np.ndim == 2:
        src_np = src_np.T
    sf.write(filepath, src_np, int(sample_rate))

torchaudio.load = patched_torchaudio_load
torchaudio.save = patched_torchaudio_save
print("✅ Patched torchaudio load/save to use soundfile backend (bypassing torchcodec)")
sys.stdout.flush()

import gradio as gr

# ==== GPU & RAM Info ====
print(f"GPU Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"VRAM Available: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB" if torch.cuda.is_available() else "VRAM: N/A")
ram = psutil.virtual_memory()
print(f"RAM Available: {ram.available / 1024**3:.1f} GB / {ram.total / 1024**3:.1f} GB")
sys.stdout.flush()

torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)

from shared.utils import files_locator as fl
from postprocessing.seedvc.wgp_bridge import convert_audio_file, _merge_audio_files_to_wav
from preprocessing.extract_vocals import extract_vocal_and_background_stems

# ==== VOICE CHANGER PROCESSOR ====
@torch.inference_mode()
def Change_Voice(source_audio, target_voice, mode_label, demix, persist, steps, cfg_rate, progress=gr.Progress()):
    try:
        if not source_audio or not target_voice:
            return None, "❌ Please provide both a source audio file (your vocal/song) and a target voice reference audio."

        progress(0, desc="Initializing conversion...")
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        # Map modes
        mode_map = {
            "v1.0 Speech": 1,
            "v1.0 Singing / F0 44k": 2,
            "v2 Speech": 3
        }
        mode = mode_map[mode_label]

        print(f"\n{'='*60}")
        print(f"Executing voice conversion:")
        print(f"  Mode: {mode_label}")
        print(f"  Source: {source_audio}")
        print(f"  Reference: {target_voice}")
        print(f"  Demixing (Background separation): {demix}")
        print(f"  Persistent in GPU: {persist}")
        print(f"  Diffusion Steps: {steps} | CFG Rate: {cfg_rate}")
        print(f"{'='*60}")
        sys.stdout.flush()

        output_dir = tempfile.mkdtemp()
        output_path = os.path.join(output_dir, "cloned_output.wav")

        temp_tracks = []
        conversion_source_path = source_audio
        background_path = None
        conversion_output_path = output_path

        # 1. Stem separation (vocals vs backing tracks)
        if demix:
            progress(0.1, desc="Isolating vocals from background music (RoFormer demix)...")
            print("  [Status] Separating Stems...")
            sys.stdout.flush()

            vocals_temp = os.path.join(output_dir, "extracted_vocals.wav")
            background_temp = os.path.join(output_dir, "extracted_background.wav")

            conversion_source_path, background_path = extract_vocal_and_background_stems(
                source_audio, vocals_temp, background_temp
            )
            temp_tracks += [conversion_source_path, background_path]

            conversion_output_path = os.path.join(output_dir, "converted_vocals.wav")
            temp_tracks.append(conversion_output_path)

        # 2. Voice conversion using SeedVC
        progress(0.4, desc="Converting vocals to target speaker voice...")
        print("  [Status] Loading model & converting voice...")
        sys.stdout.flush()

        # Set paths locator to point to models directory containing checkpoints
        fl.set_checkpoints_paths(["models", "ckpts", "."])

        convert_audio_file(
            source_audio_path=conversion_source_path,
            voice_sample_path=target_voice,
            output_path=conversion_output_path,
            persistent_models=persist,
            profile_no=4, # mmgp Profile 4
            verbose_level=1,
            diffusion_steps=int(steps),
            cfg_rate=float(cfg_rate),
            mode=mode
        )

        # 3. Remix background back in
        if background_path is not None:
            progress(0.85, desc="Merging converted vocals with original background music...")
            print("  [Status] Remixing back to final output...")
            sys.stdout.flush()
            _merge_audio_files_to_wav([conversion_output_path, background_path], output_path)
        else:
            output_path = conversion_output_path

        # Cleanup temporary files
        if temp_tracks:
            from shared.utils.audio_video import cleanup_temp_audio_files
            cleanup_temp_audio_files(temp_tracks)

        progress(1.0, desc="Success!")
        print("  ✅ Complete!")
        sys.stdout.flush()
        return output_path, f"✅ Done! Voice Conversion complete."

    except Exception as e:
        traceback.print_exc()
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        return None, f"❌ Error: {str(e)}"

# ==== Gradio Custom Styling ====
CSS = """@import url('https://fonts.googleapis.com/css2?family=Outfit:wght@400;500;600;700&display=swap');
* { font-family: 'Outfit', sans-serif !important; }
.gradio-container { max-width: 1000px !important; margin: auto !important; }
.brand-header { text-align: center; background: linear-gradient(135deg, #1e0b36 0%, #4c1d95 100%); padding: 32px; border-radius: 16px; margin-bottom: 24px; box-shadow: 0 10px 30px rgba(76,29,149,0.3); }
.brand-title { color: white; font-size: 2.2em; font-weight: 700; margin: 0 0 6px 0; }
.brand-subtitle { color: #d8b4fe; font-size: 1.1em; margin-bottom: 18px; }
.social-buttons { display: flex; justify-content: center; gap: 12px; flex-wrap: wrap; }
.social-btn { padding: 10px 24px; border-radius: 8px; font-weight: 600; font-size: 15px; text-decoration: none; display: inline-block; color: white; transition: all 0.3s; box-shadow: 0 4px 12px rgba(0,0,0,0.2); }
.social-btn:hover { transform: translateY(-2px); box-shadow: 0 6px 16px rgba(0,0,0,0.3); }
.youtube-btn { background: linear-gradient(135deg, #FF0000 0%, #CC0000 100%); }
.x-btn { background: linear-gradient(135deg, #000000 0%, #333333 100%); }
button.primary { background: linear-gradient(135deg, #7c3aed 0%, #4c1d95 100%) !important; color: white !important; font-weight: 600 !important; border-radius: 12px !important; border: none !important; }
#stop-btn { background: linear-gradient(135deg, #ef4444 0%, #b91c1c 100%) !important; color: white !important; font-weight: 600 !important; border-radius: 12px !important; border: none !important; }
#clear-btn { background: linear-gradient(135deg, #6b7280 0%, #374151 100%) !important; color: white !important; font-weight: 600 !important; border-radius: 12px !important; border: none !important; }
.footer { text-align: center; padding: 24px; margin-top: 36px; border-top: 1px solid #e5e7eb; color: #6b7280; }
"""

with gr.Blocks(css=CSS, theme=gr.themes.Soft(), title="SeedVC Voice Changer & Singing Cloner | AIQUEST") as demo:
    gr.HTML('<div class="brand-header"><div class="brand-title">🎙️ SeedVC Voice Changer & Singing Cloner</div><div class="brand-subtitle">Created by <strong>AIQuest Academy</strong> | Colab Free Tier</div><div class="social-buttons"><a href="https://youtube.com/@aiquestacademy" target="_blank" class="social-btn youtube-btn">▶️ Subscribe on YouTube</a><a href="https://x.com/aiquestacademy" target="_blank" class="social-btn x-btn">𝕏 Follow on X</a></div></div>')

    gr.Markdown(
        "Upload a source audio file (like a singing track or speech) and a target voice reference audio file. "
        "The model will replace the vocals with the target reference voice while keeping the backing track intact if **Demixing** is enabled!"
    )

    with gr.Row():
        with gr.Column(scale=1):
            source_audio = gr.Audio(label="🎵 Source Audio (Speech or Song Vocals)", type="filepath")
            target_voice = gr.Audio(label="👤 Target Speaker reference voice", type="filepath")

            mode_label = gr.Radio(
                label="Conversion Mode",
                choices=["v1.0 Speech", "v1.0 Singing / F0 44k", "v2 Speech"],
                value="v1.0 Speech"
            )

            demix = gr.Checkbox(label="Enable Vocal/Background Separation (Demixing)", value=True)
            persist = gr.Checkbox(label="Keep model persistent in GPU memory", value=True)

        with gr.Column(scale=1):
            with gr.Accordion("⚙️ Advanced Tuning Parameters", open=True):
                steps = gr.Slider(label="⚡ Diffusion Steps", minimum=5, maximum=100, step=1, value=25)
                cfg_rate = gr.Slider(label="🎯 CFG Rate (Similarity Control)", minimum=0.1, maximum=1.5, step=0.05, value=0.5)

            with gr.Row():
                gen_btn = gr.Button("🎙️ Change Voice", variant="primary", size="lg")
                stop_btn = gr.Button("🛑 Stop", variant="secondary", size="lg", elem_id="stop-btn")
                clear_btn = gr.Button("🗑️ Clear", variant="secondary", size="lg", elem_id="clear-btn")

            audio_out = gr.Audio(label="🔊 Converted Audio Output", type="filepath")
            status_out = gr.Textbox(label="Status Log", interactive=False)

    # Dynamic default parameter updates when switching modes
    def on_mode_change(mode):
        if mode == "v1.0 Speech":
            return gr.update(value=25), gr.update(value=0.5)
        elif mode == "v1.0 Singing / F0 44k":
            return gr.update(value=10), gr.update(value=0.7)
        else: # v2 Speech
            return gr.update(value=30), gr.update(value=0.7)

    mode_label.change(
        fn=on_mode_change,
        inputs=[mode_label],
        outputs=[steps, cfg_rate]
    )

    gen_event = gen_btn.click(
        fn=Change_Voice,
        inputs=[source_audio, target_voice, mode_label, demix, persist, steps, cfg_rate],
        outputs=[audio_out, status_out]
    )
    stop_btn.click(fn=None, cancels=[gen_event])
    clear_btn.click(
        fn=lambda: (None, None, "v1.0 Speech", True, True, 25, 0.5, None, ""),
        outputs=[source_audio, target_voice, mode_label, demix, persist, steps, cfg_rate, audio_out, status_out]
    )

    gr.HTML('<div class="footer"><p style="font-size: 16px; margin: 5px 0;">🎙️ Created by <strong>AIQuest Academy</strong></p><p style="font-size: 14px; margin: 5px 0; color: #9ca3af;">Free &amp; Open Source | Zero-Shot Voice Conversion | Colab Free Tier</p><p style="font-size: 13px; margin: 10px 0;"><a href="https://youtube.com/@aiquestacademy" target="_blank" style="color: #6d28d9; text-decoration: none; margin: 0 10px;">YouTube</a> | <a href="https://x.com/aiquestacademy" target="_blank" style="color: #6d28d9; text-decoration: none; margin: 0 10px;">X (Twitter)</a> | <a href="https://aiquest.site" target="_blank" style="color: #6d28d9; text-decoration: none; margin: 0 10px;">aiquest.site</a></p></div>')

print("\nLaunching Gradio...")
sys.stdout.flush()
demo.queue()
demo.launch(share=True, inline=False, debug=True, show_error=True, max_threads=1, ssr_mode=False)

In [ ]:
#@title 🚀 Step 5: Launch!
# Runs the voice changer script. Watch for the public share link in the console output.
!cd /content && python -u run_seedvc_changer.py 2>&1